In [14]:
import pandas as pd
import re
from constants.db_connections import ENGINE_READ_ONLY
import os
import paramiko



pd.set_option('display.max_rows', 700)

In [50]:

# Set your server details
hostname = 'dandyweb01fl'  # Replace with your server's IP or hostname
port = 22                       # Usually 22 for SSH
username = 'glj523'      # Replace with your username
password = 'Wtcantfw36c!123'      # Replace with your password

remote_directories = ['/datasets/caeg_fastq/2023/20230927_A00706_0773_AHNLW5DSX5_AOZCK/eDNALib060-Kurt',
                      '/datasets/caeg_fastq/2023/20230927_A00706_0773_AHNLW5DSX5_AOZCK/eDNALib060-Thorfinn',
                      '/datasets/caeg_fastq/2024/20240702_A00706_0862_BH5F5KDSX7_WBDQ4_new/ssDNALib0019']

lib_ids_all = {}

remote_directory = remote_directories[0]  # Replace with the directory you want to list


# Create an SSH client
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    # Connect to the server
    ssh.connect(hostname, port, username, password)

    # Run the command to list files and directories

    for dir in remote_directories:
    
        stdin, stdout, stderr = ssh.exec_command(f"ls {dir} | grep '\.fastq.*$'")

        _, test, _ = ssh.exec_command(f"ls {dir} | grep '\.fastq.*$' | wc -l")
    

        # Process the output
        file_names = stdout.read().decode().splitlines()
        test = test.read().decode()

        lib_ids = [file_name.split("-")[0] for file_name in file_names]

        if int(test) != len(lib_ids):
            raise Exception("Error")

        lib_ids_all[dir.split("/")[-1]] = list(set(lib_ids))
    

        # Print the first 8 letters of each file/directory name
   

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # Close the connection
    ssh.close()


In [13]:
df = pd.read_sql(sql='select * from test_1.mega_table_qc_mat', con=ENGINE_READ_ONLY)

In [53]:
test_lib = lib_ids_all["eDNALib060-Kurt"][0]

In [ ]:
lib_ids_all["eDNALib060-Kurt"][0]

LV7009026434


In [57]:
df.head()

,sample_name,bowtie2__overall_alignment_rate,bowtie2__total_reads,bowtie2__unpaired_aligned_multi,bowtie2__unpaired_aligned_none,bowtie2__unpaired_aligned_one,bowtie2__unpaired_total,fastp__adapter_cutting,fastp__command,fastp__filtering_result,...,samtools_stats__reads_properly_paired_percent,samtools_stats__reads_QC_failed,samtools_stats__reads_QC_failed_percent,samtools_stats__reads_unmapped,samtools_stats__reads_unmapped_percent,samtools_stats__sequences,samtools_stats__supplementary_alignments,samtools_stats__total_first_fragment_length,samtools_stats__total_last_fragment_length,samtools_stats__total_length
0,Lib_LV3000765846_collapsed,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,...,0.0,0.0,0.0,0.0,0.0,6717886.0,0.0,432800085.0,0.0,432800085.0
1,Lib_LV3000765846_collapsed.norway.1-of-7,0.04,147126867.0,36910.0,147068549.0,21408.0,147126867.0,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Lib_LV3000765846_collapsed.norway.2-of-7,0.02,147126867.0,22432.0,147091204.0,13231.0,147126867.0,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Lib_LV3000765846_collapsed.norway.3-of-7,0.02,147126867.0,21759.0,147091762.0,13346.0,147126867.0,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lib_LV3000765846_collapsed.norway.4-of-7,0.03,147126867.0,24393.0,147088860.0,13614.0,147126867.0,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
for e in lib_ids_all["eDNALib060-Kurt"][0]:
    print(df[df["sample_name"].str.contains(e)])

                                    sample_name  \
0                    Lib_LV3000765846_collapsed   
1      Lib_LV3000765846_collapsed.norway.1-of-7   
2      Lib_LV3000765846_collapsed.norway.2-of-7   
3      Lib_LV3000765846_collapsed.norway.3-of-7   
4      Lib_LV3000765846_collapsed.norway.4-of-7   
...                                         ...   
92037                     Lib_LV7009027909_L004   
92038           Lib_LV7009027909_L004_collapsed   
92039                  Lib_LV7009027909_L004_R1   
92040                  Lib_LV7009027909_L004_R2   
92041           Lib_LV7009027909_L004_singleton   

       bowtie2__overall_alignment_rate  bowtie2__total_reads  \
0                                  NaN                   NaN   
1                                 0.04           147126867.0   
2                                 0.02           147126867.0   
3                                 0.02           147126867.0   
4                                 0.03           147126867.0   
...